In [2]:
%load_ext jupyter_ai

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [4]:
%%ai openai-chat:gemma-3-27b-it
Hello — just reply with the single word GEMMA_OK

GEMMA_OK


# 1. Debugging

The model should find and fix a bug, explain the cause, and prove the fix with a quick run. Execute the buggy code below.

In [ ]:
def moving_average(xs, window):
    """
    Return the moving average over `window` elements.
    Example: xs=[1,2,3,4], window=2 -> [1.5, 2.5, 3.5]
    """
    avgs = []
    for i in range(len(xs)):
        chunk = xs[i:i+window]
        print(chunk)
        avgs.append(sum(chunk) / window)  # bug: divides by window even at end
    return avgs

print(moving_average([1,2,3,4], 2))
# Expected: [1.5, 2.5, 3.5]

### Use %%ai magic to fix the bug

Now we can invoke the %%ai magic and prompt the model to identify the bug, fix the code, and add a sanity check just to make doubly certain that the fix works.

**What a “good” fix looks like**
- The model correctly identifies the bug. In this case, it notices that the last notices the last slice(s) are shorter than window.
- The fix preserves the expected behavior specified in the docstring. 
- A sanity check is provided.

In [ ]:
%%ai openai-chat:gemma-3-27b-it

The output is wrong. Identify the bug and fix it. Keep behavior exactly as described in the docstring. Add one quick sanity check.

def moving_average(xs, window):
    """
    Return the moving average over `window` elements.
    Example: xs=[1,2,3,4], window=2 -> [1.5, 2.5, 3.5]
    """
    avgs = []
    for i in range(len(xs)):
        chunk = xs[i:i+window]
        avgs.append(sum(chunk) / window)  # bug: divides by window even at end
    return avgs

print(moving_average([1,2,3,4], 2))
# Expected: [1.5, 2.5, 3.5]

### Test the corrected code and sanity check

In the empty cells below, paste the model solution and sanity check. Run them and ensure that the result is indeed what is expected.

In [ ]:
# Paste and run the fixed code here

In [ ]:
# Paste and run the new sanity check here

# 2. Writing test cases

Below is a function that is meant to convert any string into a slug that can be used in a custom URL. Currently, it fails when there are multiple spaces (for example, it allows for the generation of hello--world). Try running the code yourself.

In [ ]:
def slugify(text: str) -> str:
    """Convert text into a URL-friendly slug."""
    return (
        text.strip()
            .lower()
            .replace(" ", "-")
            .replace("--", "-")
    )

# Try a few manually:
print(slugify(" Hello World "))
print(slugify("Hello   World"))

### Using Jupyter AI to write test cases

Now we can run the cell again, but add a prompt at the top. In the prompt, we will request that the AI generate at least 3 edge cases. It should definitely come up with a test that demonstrates the failure with multiple spaces. Some other good edge cases that it should test include
- Leading/trailing spaces
- Already-hyphenated input
- Empty string / whitespace-only string

In [ ]:
%%ai openai-chat:gemma-3-27b-it

Write pytest tests for this function. Cover normal case + at least 3 edge cases. Don’t change the function yet.

def slugify(text: str) -> str:
    """Convert text into a URL-friendly slug."""
    return (
        text.strip()
            .lower()
            .replace(" ", "-")
            .replace("--", "-")
    )

# Try a few manually:
print(slugify(" Hello  World "))

Go ahead and paste the AI-generated Pytest tests in the empty cell below, then run it.

In [5]:
# Paste and run the AI-generated tests here

Now run the cell below to check all of the tests that Jupyter AI came up with. The one that tests for multiple spaces should fail. 

In [ ]:
# Check if any of the tests fail.

def run_all_tests():
    failures = 0
    for name, obj in globals().items():
        if name.startswith("test_") and callable(obj):
            try:
                obj()
            except AssertionError:
                print(f"{name} FAILED")
                failures += 1
    if failures == 0:
        print("All tests passed!")
    else:
        print(f"{failures} tests failed.")

run_all_tests()

### Update slugify()

If you would like, you can ask the model to update the slugify() function so that it passes all of the tests that it generated.

In [ ]:
%%ai gemini:gemini-2.5-flash

Now update slugify() so the tests pass, but keep it simple and readable.

Paste the updated function in the empty cell below and run it.

In [ ]:
# Paste and run the updated slugify() function here

Confirm that it can now handle multiple spaces by running it with the same inputs as earlier.

In [ ]:
print(slugify(" Hello World "))
print(slugify("Hello   World"))

You can also confirm that the Pytests tests pass by rerunning the cell with the ```run_all_tests()``` function

# 3. Adding a function

The model should be able to add a function with clear requirements, integrate it, and demonstrate that it works. Below we have a function that splits an input string into a list of word tokens

In [ ]:
from collections import Counter
import re

def tokenize(text: str) -> list[str]:
    """Split text into lowercase word tokens."""
    return re.findall(r"[a-zA-Z']+", text.lower())

print(tokenize("All your base are belong to us."))

### Generate a new function

Now we'll use the %%ai cell magic to create a new function that does something simple but easily demonstrates the model's capability. Below, we prompt it to create ```top_k_words()```. A well-performing model will check the following boxes:
- Follows the prompt exactly
- Doesn't rewrite ```tokenize()```
- The example provides a good test of the logic 

In [ ]:
%%ai openai-chat:gemma-3-27b-it

Add top_k_words(text, k) under tokenize. 
It should return a list of (word, count), sorted by count desc, then word asc. 
Ignore words shorter than 3 chars. Show a short example and output.

from collections import Counter
import re

def tokenize(text: str) -> list[str]:
    """Split text into lowercase word tokens."""
    return re.findall(r"[a-zA-Z']+", text.lower())

### Test the new function and example

Now you should make sure that the model's newly created function works as expected. Paste and run the new function (and any imports), plus the provided example, in the cells below.

In [ ]:
# Paste and run the new function code here

In [ ]:
# Paste and run the example here

# 4. Generating docstrings

We want the model to be able to generate detailed and accurate docstrings for a function. It should match the behavior of the function, correctly identify variable types, identify any errors raised, and provide good examples. Below is the ```clip()``` function. Run it to see how it works.

In [ ]:
def clip(values, lo, hi):
    out = []
    for v in values:
        if v < lo:
            out.append(lo)
        elif v > hi:
            out.append(hi)
        else:
            out.append(v)
    return out

# Test case
print(clip([1, 5, 10], 2, 8))  # Expected result is [2, 5, 8]

### Let Jupyter AI come up with the docstring

In the next cell, we will prompt the model to give us a docstring in NumPy/Google style. A good docstring ticks the following boxes:
- Accurately describes the types for ```values```, ```lo```, and ```hi```. 
- Does not fabricate exceptions that aren't actually raised by the code
- Contains runnable examples that match the expected output

In [ ]:
%%ai openai-chat:gemma-3-27b-it

Write a high-quality docstring in NumPy style (or Google style) for the clip() function. 
The docstring should match the current behavior exactly. 
Include Args, Returns, Raises (if any), and 1-2 examples.

def clip(values, lo, hi):
    out = []
    for v in values:
        if v < lo:
            out.append(lo)
        elif v > hi:
            out.append(hi)
        else:
            out.append(v)
    return out

print(clip([1, 5, 10], 2, 8))  # [2, 5, 8]

### Example checking

Feel free to use the below cell to run the examples that the model should have generated in the docstring.

In [ ]:
# Paste and run examples here

# 5. Explaining code

One of the main functions of an AI assistant should be to explain code in plain language. But we don't want it to just paraphrase, line-by-line, what the code is doing. We want to see the model demonstrate **understanding**. Below is the function ```paths()```, and though it is short, there are several opportunities for the AI to show off its chops. Look it over and then execute the cell to see the output of the print statement.

In [ ]:
def paths(m, n, memo=None):
    if memo is None:
        memo = dict()
    if m == 1 or n == 1:
        return 1
    if (m, n) in memo:
        return memo[(m, n)]
    memo[(m, n)] = paths(m-1, n, memo) + paths(m, n-1, memo)
    return memo[(m, n)]

print(paths(3, 4))

### Request an explanation

In explaining code, model responses should tick the following boxes:
- Explains the actual **purpose** of the code (without hallucinating)
- Identifies key concepts (i.e., "This uses memoization," "This is a common pitfall," etc.)
- Doesn't just state what the code is doing, but explains why it is doing it
- Predicts the output correctly
- Points out subtleties or improvements

Run the cell below to see how well the AI model attached to Jupyter AI explains the code.

In [ ]:
%%ai openai-chat:gemma-3-27b-it

Explain what the code below is doing

def paths(m, n, memo=None):
    if memo is None:
        memo = dict()
    if m == 1 or n == 1:
        return 1
    if (m, n) in memo:
        return memo[(m, n)]
    memo[(m, n)] = paths(m-1, n, memo) + paths(m, n-1, memo)
    return memo[(m, n)]

print(paths(3, 4))

# 6. Refactoring code

When we ask the model to refactor code, we are looking for it to improve clarity, reduce structural complexity, and better follow the style of the programming language of choice (in this case, Python). Below is the function ```classify_score()```, which uses excessive nesting.

In [ ]:
def classify_score(score):
    if score >= 90:
        return "A"
    else:
        if score >= 80:
            return "B"
        else:
            if score >= 70:
                return "C"
            else:
                if score >= 60:
                    return "D"
                else:
                    return "F"

### Refactor with Jupyter AI magics

Below, we prompt the model to refactor this function. Look for the following in a good model output:
- Improves readability (removes nesting)
- Doesn't change behavior (keeps grading thresholds the same)
- Follows instructions (only returns the updated function)
- Avoids over-engineering (doesn't add a dictionary) 

In [ ]:
%%ai openai-chat:gemma-3-27b-it

Refactor classify_score() to improve readability without changing behavior.
Return only the refactored function.

def classify_score(score):
    if score >= 90:
        return "A"
    else:
        if score >= 80:
            return "B"
        else:
            if score >= 70:
                return "C"
            else:
                if score >= 60:
                    return "D"
                else:
                    return "F"